In [ ]:
!pip install -q --upgrade torchao

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm
import torch
import torch.nn.functional as F
import math

In [ ]:
from datasets import load_dataset

ds = load_dataset("lmms-lab/COCO-Caption")
print(ds)

DatasetDict({
    val: Dataset({
        features: ['question_id', 'image', 'question', 'answer', 'id', 'license', 'file_name', 'coco_url', 'height', 'width', 'date_captured'],
        num_rows: 40504
    })
    test: Dataset({
        features: ['question_id', 'image', 'question', 'answer', 'id', 'license', 'file_name', 'coco_url', 'height', 'width', 'date_captured'],
        num_rows: 40775
    })
})


### Vision Encoder!!

In [ ]:
from transformers import ViTImageProcessor, ViTModel
img_processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
img_model = ViTModel.from_pretrained('google/vit-base-patch16-224')

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from peft import LoraConfig, get_peft_model


target_modules = [
    "query",
    "value",
    "key",
    "intermediate.dense"
]

lora_config = LoraConfig(
    r = 32,
    lora_alpha = 32,
    target_modules = target_modules,
    lora_dropout = 0.1,
    bias = "none"
)
img_encoder = get_peft_model(img_model,lora_config)
img_encoder.print_trainable_parameters()

trainable params: 3,244,032 || all params: 89,633,280 || trainable%: 3.6192


### text encoder!

In [ ]:
from transformers import BertTokenizer,BertModel
tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased")
text_encoder = BertModel.from_pretrained("google-bert/bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import random

class CocoDataset(Dataset):
    def __init__(self, ds, processor, tokenizer, max_length=20):
        self.ds = ds
        self.processor = processor
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return self.ds.num_rows

    def __getitem__(self, idx):
        sample = self.ds[idx]
        img = sample['image']

        if img.mode != 'RGB':
            img = img.convert('RGB')

        captions = sample['answer']
        caption = random.choice(captions)
        caption = caption.strip()

        text_tokens = self.tokenizer(
            caption,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        processed_img = self.processor(
            images=[img],
            return_tensors="pt"
        )

        return {
            'pixel_values': processed_img['pixel_values'].squeeze(0),
            'input_ids': text_tokens['input_ids'].squeeze(0),
            'attention_mask': text_tokens['attention_mask'].squeeze(0)
        }

In [ ]:
train_dataset = CocoDataset(ds['val'].select(range(40000)),img_processor,tokenizer)
test_dataset = CocoDataset(ds['val'].select(range(40000,40504)),img_processor,tokenizer)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2)
test_loader = DataLoader(test_dataset,batch_size=32,num_workers=2)

In [ ]:
import torch.nn as nn
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm
import torch.nn.functional as F
import math
from dataclasses import dataclass, field
from typing import Tuple

@dataclass
class StageIConfig:
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    epochs: int = 10
    batch_size: int = 32

    lr: float = 3e-4
    weight_decay: float = 1e-2
    betas: Tuple[float, float] = field(default_factory=lambda: (0.9, 0.999))
    eps: float = 1e-8
    max_grad_norm: float = 1.0

    pct_start: float = 0.1
    anneal_strategy: str = 'cos'

    max_length: int = 128
    save_path: str = 'best_img_encoder.pt'


class StageITraining():
    def __init__(self, img_encoder, text_encoder, train_loader, test_loader, training_config):
        self.cfg = training_config
        self.img_encoder = img_encoder.to(self.cfg.device)
        self.text_encoder = text_encoder.to(self.cfg.device)
        self.train_loader = train_loader
        self.test_loader = test_loader

        self.temp = nn.Parameter(
            torch.ones([], device=self.cfg.device) * math.log(10.0)
        )

        trainable_params = [
            p for p in self.img_encoder.parameters() if p.requires_grad
        ] + [self.temp]

        self.optimizer = AdamW(
            trainable_params,
            lr=self.cfg.lr,
            weight_decay=self.cfg.weight_decay,
            betas=self.cfg.betas,
            eps=self.cfg.eps
        )
        self.scheduler = OneCycleLR(
            self.optimizer,
            max_lr=self.cfg.lr,
            steps_per_epoch=len(self.train_loader),
            epochs=self.cfg.epochs,
            pct_start=self.cfg.pct_start,
            anneal_strategy=self.cfg.anneal_strategy,
        )

    def sigmoid_loss(self, img_vec, text_vec):
        temp = torch.clamp(self.temp.exp(), min=1.0, max=100.0)
        logits = img_vec @ text_vec.T * temp

        n = logits.shape[0]
        labels = 2 * torch.eye(n, device=logits.device) - 1
        loss = -F.logsigmoid(labels * logits).mean()
        return loss

    def fit(self):
        scaler = torch.amp.GradScaler('cuda')
        best_val_loss = float('inf')

        for ep in range(self.cfg.epochs):

            self.img_encoder.train()
            self.text_encoder.eval()

            train_loss = 0.0
            pbar = tqdm(self.train_loader, desc=f"Epoch {ep+1}/{self.cfg.epochs}")

            for step, batch in enumerate(pbar):
                pixel_values = batch['pixel_values'].to(self.cfg.device)
                input_ids = batch['input_ids'].to(self.cfg.device)
                attention_mask = batch['attention_mask'].to(self.cfg.device)

                self.optimizer.zero_grad()

                with torch.amp.autocast('cuda'):
                    img_vec = self.img_encoder(pixel_values).last_hidden_state[:, 0, :]
                    with torch.no_grad():
                        text_vec = self.text_encoder(input_ids, attention_mask).last_hidden_state[:, 0, :]

                    img_vec = F.normalize(img_vec, dim=-1)
                    text_vec = F.normalize(text_vec, dim=-1)

                    loss = self.sigmoid_loss(img_vec, text_vec)

                scaler.scale(loss).backward()
                scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in self.img_encoder.parameters() if p.requires_grad] + [self.temp],
                    self.cfg.max_grad_norm
                )
                scaler.step(self.optimizer)
                scaler.update()
                self.scheduler.step()

                train_loss += loss.item()
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'lr': f'{self.scheduler.get_last_lr()[0]:.6f}',
                    'temp': f'{self.temp.exp().item():.3f}'
                })

            avg_train_loss = train_loss / len(self.train_loader)
            val_loss = self.evaluate()
            print(f"Epoch {ep+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Temp: {self.temp.exp().item():.3f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save({
                    'img_encoder': self.img_encoder.state_dict(),
                    'temp': self.temp.data,
                }, self.cfg.save_path)
                print(f"Best model saved at epoch {ep+1}")

    @torch.no_grad()
    def evaluate(self):
        self.img_encoder.eval()
        self.text_encoder.eval()

        total_loss = 0.0
        for batch in tqdm(self.test_loader, desc="Validating"):
            pixel_values = batch['pixel_values'].to(self.cfg.device)
            input_ids = batch['input_ids'].to(self.cfg.device)
            attention_mask = batch['attention_mask'].to(self.cfg.device)

            with torch.amp.autocast('cuda'):
                img_vec = self.img_encoder(pixel_values).last_hidden_state[:, 0, :]
                text_vec = self.text_encoder(input_ids, attention_mask).last_hidden_state[:, 0, :]

                img_vec = F.normalize(img_vec, dim=-1)
                text_vec = F.normalize(text_vec, dim=-1)

                loss = self.sigmoid_loss(img_vec, text_vec)

            total_loss += loss.item()

        return total_loss / len(self.test_loader)

In [ ]:
cfg = StageIConfig()
print(cfg)

StageIConfig(device='cuda', epochs=10, batch_size=32, lr=0.0003, weight_decay=0.01, betas=(0.9, 0.999), eps=1e-08, max_grad_norm=1.0, pct_start=0.1, anneal_strategy='cos', max_length=128, save_path='best_img_encoder.pt')


In [ ]:
trainer = StageITraining(img_encoder, text_encoder, train_loader, test_loader, cfg)

In [ ]:
trainer.fit()

Validating: 100%|██████████| 16/16 [00:04<00:00,  3.42it/s]


Epoch 1 | Train Loss: 0.2134 | Val Loss: 0.1323 | Temp: 10.269
  ✅ Best model saved at epoch 1


Validating: 100%|██████████| 16/16 [00:05<00:00,  3.07it/s]


Epoch 2 | Train Loss: 0.1155 | Val Loss: 0.1282 | Temp: 12.559
  ✅ Best model saved at epoch 2


Validating: 100%|██████████| 16/16 [00:04<00:00,  3.49it/s]


...

Epoch 8 | Train Loss: 0.0657 | Val Loss: 0.1158 | Temp: 39.545
  ✅ Best model saved at epoch 8


Validating: 100%|██████████| 16/16 [00:05<00:00,  3.00it/s]


Epoch 9 | Train Loss: 0.0638 | Val Loss: 0.1274 | Temp: 40.479


Validating: 100%|██████████| 16/16 [00:05<00:00,  2.84it/s]

Epoch 10 | Train Loss: 0.0632 | Val Loss: 0.1238 | Temp: 40.620
